# Build Master Dataset (Notebook Version)

This notebook mirrors `src/build_master_dataset.py`.

## Inputs (under `data/raw/`)
- `FAOSTAT_data_en_4-29-2026.csv`
- `owid-co2-data.csv`
- `owid-energy-data.csv`

## Output (written to `data/processed/`)
- `final_merged.csv`

Run cells in order from top to bottom. If you open Jupyter with the working directory set to either the repo root or `notebooks/`, the paths below resolve automatically.

In [ ]:
import pandas as pd
from pathlib import Path


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "raw").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "data" / "raw").is_dir():
        return cwd.parent
    return cwd


REPO_ROOT = _repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

# STEP 0: File paths
faostat_path = DATA_RAW / "FAOSTAT_data_en_4-29-2026.csv"
co2_path = DATA_RAW / "owid-co2-data.csv"
energy_path = DATA_RAW / "owid-energy-data.csv"
output_path = DATA_PROCESSED / "final_merged.csv"

for p in [faostat_path, co2_path, energy_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {p}")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print("All input files found.")

In [ ]:
# STEP 1: Initial data inspection
faostat_raw = pd.read_csv(faostat_path)
co2_raw = pd.read_csv(co2_path)
energy_raw = pd.read_csv(energy_path)

datasets = {
    "FAOSTAT": faostat_raw,
    "OWID CO2": co2_raw,
    "OWID ENERGY": energy_raw,
}

for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(f"Shape: {df.shape}")
    print("Columns:")
    print(list(df.columns))
    print("\nDtypes:")
    print(df.dtypes)
    print("\nSample rows:")
    print(df.head(3))

In [ ]:
# STEP 2: Standardize country names
faostat_raw["country"] = faostat_raw["Area"].astype(str).str.strip()
co2_raw["country"] = co2_raw["country"].astype(str).str.strip()
energy_raw["country"] = energy_raw["country"].astype(str).str.strip()

country_mapping = {
    "United States of America": "United States",
    "Viet Nam": "Vietnam",
    "Iran (Islamic Republic of)": "Iran",
    "Russian Federation": "Russia",
    "Czechia": "Czech Republic",
    "Republic of Korea": "South Korea",
    "Türkiye": "Turkey",
    "United Republic of Tanzania": "Tanzania",
    "Syrian Arab Republic": "Syria",
    "Bolivia (Plurinational State of)": "Bolivia",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Democratic Republic of the Congo": "Democratic Republic of Congo",
    "Congo": "Republic of Congo",
    "Lao People's Democratic Republic": "Laos",
    "China, Taiwan Province of China": "Taiwan",
    "China, Hong Kong SAR": "Hong Kong",
    "China, Macao SAR": "Macao",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "State of Palestine": "Palestine",
    "Bahamas": "The Bahamas",
    "Gambia": "The Gambia",
}

faostat_raw["country"] = faostat_raw["country"].replace(country_mapping)
co2_raw["country"] = co2_raw["country"].replace(country_mapping)
energy_raw["country"] = energy_raw["country"].replace(country_mapping)

faostat_countries = set(faostat_raw["country"].unique())
co2_countries = set(co2_raw["country"].unique())
energy_countries = set(energy_raw["country"].unique())
common_countries = faostat_countries.intersection(co2_countries).intersection(energy_countries)

print(f"Countries in FAOSTAT: {len(faostat_countries)}")
print(f"Countries in OWID CO2: {len(co2_countries)}")
print(f"Countries in OWID ENERGY: {len(energy_countries)}")
print(f"Common countries across all datasets: {len(common_countries)}")

In [ ]:
# STEP 3: Align time dimension
faostat_raw["year"] = pd.to_numeric(faostat_raw["Year"], errors="coerce").astype("Int64")
co2_raw["year"] = pd.to_numeric(co2_raw["year"], errors="coerce").astype("Int64")
energy_raw["year"] = pd.to_numeric(energy_raw["year"], errors="coerce").astype("Int64")

faostat_years = set(faostat_raw["year"].dropna().astype(int).unique())
co2_years = set(co2_raw["year"].dropna().astype(int).unique())
energy_years = set(energy_raw["year"].dropna().astype(int).unique())
common_years = sorted(list(faostat_years.intersection(co2_years).intersection(energy_years)))

print(f"FAOSTAT year range: {min(faostat_years)} to {max(faostat_years)}")
print(f"OWID CO2 year range: {min(co2_years)} to {max(co2_years)}")
print(f"OWID ENERGY year range: {min(energy_years)} to {max(energy_years)}")
print(f"Common year range: {min(common_years)} to {max(common_years)}")
print(f"Number of common years: {len(common_years)}")

faostat_raw = faostat_raw[faostat_raw["year"].isin(common_years)].copy()
co2_raw = co2_raw[co2_raw["year"].isin(common_years)].copy()
energy_raw = energy_raw[energy_raw["year"].isin(common_years)].copy()

In [ ]:
# STEP 4: Temperature filtering (FAOSTAT)
temp_df = faostat_raw[
    (faostat_raw["Element"] == "Temperature change")
    & (faostat_raw["Months"] == "Meteorological year")
].copy()

temp_df["temperature_change_c"] = pd.to_numeric(temp_df["Value"], errors="coerce")
temp_df = temp_df[["country", "year", "temperature_change_c"]].copy()
temp_df = temp_df.groupby(["country", "year"], as_index=False)["temperature_change_c"].mean()

print(f"Filtered temperature dataset shape: {temp_df.shape}")
print(temp_df.head(5))

In [ ]:
# STEP 5: Merge datasets
energy_keep_cols = [
    "country", "year", "iso_code", "population", "gdp",
    "electricity_demand_per_capita", "electricity_demand", "electricity_generation",
    "energy_per_capita", "energy_per_gdp", "fossil_share_elec", "renewables_share_elec",
    "net_elec_imports_share_demand",
]

co2_keep_cols = [
    "country", "year", "iso_code", "co2", "co2_per_capita", "co2_per_gdp",
    "co2_per_unit_energy", "consumption_co2", "consumption_co2_per_capita",
    "temperature_change_from_co2", "temperature_change_from_ghg",
]

energy_df = energy_raw[energy_keep_cols].copy()
co2_df = co2_raw[co2_keep_cols].copy()

rows_energy_before = len(energy_df)
rows_co2_before = len(co2_df)
rows_temp_before = len(temp_df)

merged_ec = pd.merge(
    energy_df,
    co2_df,
    on=["country", "year"],
    how="inner",
    suffixes=("_energy", "_co2"),
)

merged_all = pd.merge(
    merged_ec,
    temp_df,
    on=["country", "year"],
    how="inner",
)

print(f"Rows in energy before merge: {rows_energy_before}")
print(f"Rows in co2 before merge: {rows_co2_before}")
print(f"Rows in temp before merge: {rows_temp_before}")
print(f"Rows after energy + co2 merge: {len(merged_ec)}")
print(f"Rows after adding temperature: {len(merged_all)}")

In [ ]:
# STEP 6: Data quality checks
missing_summary = merged_all.isna().sum().sort_values(ascending=False)
missing_pct = (merged_all.isna().mean() * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({
    "missing_count": missing_summary,
    "missing_pct": missing_pct.round(2),
})

duplicates_count = merged_all.duplicated(subset=["country", "year"]).sum()
retention_vs_energy = (len(merged_all) / rows_energy_before) * 100 if rows_energy_before > 0 else 0
retention_vs_ec = (len(merged_all) / len(merged_ec)) * 100 if len(merged_ec) > 0 else 0

print("Top columns by missing values:")
print(missing_report.head(15))
print(f"\nDuplicate country-year rows: {duplicates_count}")
print(f"Retention vs ENERGY base: {retention_vs_energy:.2f}%")
print(f"Retention vs ENERGY+CO2 stage: {retention_vs_ec:.2f}%")

In [ ]:
# STEP 7: Final cleaning
if "iso_code_energy" in merged_all.columns and "iso_code_co2" in merged_all.columns:
    merged_all["iso_code"] = merged_all["iso_code_energy"].combine_first(merged_all["iso_code_co2"])
    merged_all.drop(columns=["iso_code_energy", "iso_code_co2"], inplace=True)

final_df = merged_all[merged_all["electricity_demand_per_capita"].notna()].copy()

numeric_cols = [
    "year", "population", "gdp", "electricity_demand_per_capita", "electricity_demand",
    "electricity_generation", "energy_per_capita", "energy_per_gdp", "fossil_share_elec",
    "renewables_share_elec", "net_elec_imports_share_demand", "co2", "co2_per_capita",
    "co2_per_gdp", "co2_per_unit_energy", "consumption_co2", "consumption_co2_per_capita",
    "temperature_change_from_co2", "temperature_change_from_ghg", "temperature_change_c",
]

for col in numeric_cols:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

final_df["year"] = final_df["year"].astype(int)
final_df = final_df.sort_values(["country", "year"]).reset_index(drop=True)

print("Final column list:")
print(list(final_df.columns))
print(f"\nFinal dataset shape: {final_df.shape}")
print("\nFinal sample rows:")
print(final_df.head(10))

In [ ]:
# STEP 8: Save output
final_df.to_csv(output_path, index=False)
print(f"Saved cleaned master dataset to: {output_path.resolve()}")

print("\nASSUMPTIONS USED")
print("1) FAOSTAT 'Temperature change' at 'Meteorological year' is treated as annual climate signal.")
print("2) Inner joins are used to keep only country-year rows present across all three sources.")
print("3) Electricity-demand modeling readiness requires non-null electricity_demand_per_capita.")
print("4) Country harmonization uses a manual mapping for common naming differences.")
print("\nBalanced-panel exports (usable data, 2001-2022): run src/build_balanced_alternatives.py or notebooks/build_balanced_alternatives.ipynb")
print("- data/processed/final_merged_balanced_adjusted.csv")
print("- data/processed/final_merged_partial_balanced.csv")